<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلف: [يوري كاشنيتسكي](https://yorko.github.io). تمت الترجمة والتحرير بواسطة [سيرج أوريشكوف](https://www.linkedin.com/in/sergeoreshkov/)، و[يوانيوان باو](https://www.linkedin.com/in/yuanyuanpao/). تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.
# <center> الموضوع 8. Vowpal Wabbit: التعلم باستخدام غيغابايت من البيانات
سنغطي هذا الأسبوع سببين لسرعة تدريب Vowpal Wabbit الاستثنائية، وهما التعلم عبر الإنترنت وخدعة التجزئة، من الناحية النظرية والتطبيقية. سنجرب ذلك من خلال الأخبار ومراجعات الأفلام وأسئلة StackOverflow.



# الخطوط العريضة
1. [نسب التدرج العشوائي والتعلم عبر الإنترنت](#1.-نسب التدرج العشوائي والتعلم عبر الإنترنت)
    - 1.1. [SGD](#1.1.-نسب التدرج العشوائي)
    - 1.2. [منهج التعلم عبر الإنترنت](#1.2.-منهج التعلم عبر الإنترنت)
2. [معالجة الميزات الفئوية](#2.-معالجة الميزات الفئوية)
    - 2.1. [ترميز التسمية](#2.1.-ترميز التسمية)
    - 2.2. [ترميز سريع واحد](#2.2.-ترميز سريع واحد)
    - 2.3. [خدعة التجزئة](#2.3.-خدعة التجزئة)
3. [فووبال وابيت](#3.-فوبال-وابيت)
    - 3.1. [أخبار. التصنيف الثنائي](#3.1.-الأخبار.-التصنيف الثنائي)
    - 3.2. [أخبار. تصنيف متعدد الفئات](#3.2.-الأخبار.-تصنيف متعدد الفئات)
    - 3.3. [مراجعات أفلام IMDB](#3.3.-IMDB-مراجعات الأفلام)
    - 3.4. [تصنيف الجيجابايت من أسئلة StackOverflow](#3.4.-تصنيف الجيجابايت من أسئلة StackOverflow)
4. [مهمة تجريبية](#4.-مهمة تجريبية)
5. [موارد مفيدة](#5.-موارد-مفيدة)


In [ ]:
import os
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.datasets import fetch_20newsgroups, load_files
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, log_loss, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from tqdm import tqdm_notebook

%matplotlib inline
import seaborn as sns

In [ ]:
warnings.filterwarnings("ignore")


## 1. النسب التدرج العشوائي والتعلم عبر الإنترنت
### 1.1. نزول التدرج العشوائيعلى الرغم من حقيقة أن النسب المتدرج هو أحد الأشياء الأولى التي يتم تعلمها في دورات التعلم الآلي والتحسين، إلا أنه من الصعب التغلب على أحد تعديلاته، وهو Stochastic Gradient Descent (SGD).
تذكر أن فكرة الهبوط المتدرج هي تقليل بعض الوظائف عن طريق اتخاذ خطوات صغيرة في اتجاه الانخفاض الأسرع. تم تسمية هذه الطريقة بسبب الحقيقة التالية من حساب التفاضل والتكامل: يشير المتجه $\nabla f = (\frac{\partial f}{\partial x_1}, \ldots \frac{\partial f}{\partial x_n})^\text{T}$ للمشتقات الجزئية للدالة $f(x) = f(x_1, \ldots x_n)$ إلى اتجاه أسرع نمو للدالة. ويعني ذلك أنه من خلال التحرك في الاتجاه المعاكس (مضاد التدرج)، من الممكن تقليل قيمة الدالة بأسرع معدل.
<img src='@@KEEP_00043@@ width=50%>
إليكم متزلج على الجليد (أنا) في شيريجيش، المنتجع الشتوي الأكثر شعبية في روسيا. (أوصي به بشدة إذا كنت تحب التزلج أو التزلج على الجليد). بالإضافة إلى الإعلان عن المناظر الطبيعية الجميلة، تصور هذه الصورة فكرة النسب المتدرج. إذا كنت ترغب في الركوب بأسرع ما يمكن، فأنت بحاجة إلى اختيار المسار الأكثر انحدارًا. يمكن اعتبار حساب مضادات التدرج بمثابة تقييم للمنحدر في نقاط مختلفة.



**مثال**
يمكن حل مشكلة الانحدار المقترن بالنسب المتدرج. دعونا نتنبأ بمتغير واحد باستخدام متغير آخر: الطول مع الوزن. افترض أن هذه المتغيرات تعتمد خطيا. سوف نستخدم مجموعة البيانات [SOCR](http://wiki.stat.ucla.edu/socr/index.php/SOCR_Data). 


In [ ]:
PATH_TO_ALL_DATA = "../../data/"
data_demo = pd.read_csv(os.path.join(PATH_TO_ALL_DATA, "weights_heights.csv"))

In [ ]:
plt.scatter(data_demo["Weight"], data_demo["Height"])
plt.xlabel("Weight in lb")
plt.ylabel("Height in inches");


لدينا هنا متجه $x$ للبعد $\ell$ (وزن كل شخص، أي عينة تدريب) و$y$، وهو متجه يحتوي على ارتفاع كل شخص في مجموعة البيانات.المهمة هي التالية: العثور على الأوزان $w_0$ و$w_1$ بحيث يتم التنبؤ بالارتفاع مثل $y_i = w_0 + w_1 x_i$ (حيث $y_i$ هي $i$ - قيمة الارتفاع، $x_i$ هي $i$- قيمة الوزن) تقلل من الخطأ التربيعي (بالإضافة إلى متوسط الخطأ التربيعي نظرًا لأن $\frac{1}{\ell}$ لا يحدث أي فرق):
$$SE(w_0, w_1) = \frac{1}{2}\sum_{i=1}^\ell(y_i - (w_0 + w_1x_{i}))^2 \rightarrow min_{w_0,w_1}$$
سوف نستخدم النسب المتدرج، باستخدام المشتقات الجزئية $SE(w_0, w_1)$ على الأوزان $w_0$ و$w_1$.
يتم بعد ذلك تحديد إجراء التدريب التكراري من خلال صيغ تحديث بسيطة (نقوم بتغيير أوزان النموذج بخطوات صغيرة، بما يتناسب مع ثابت صغير $\eta$، باتجاه التدرج العكسي للدالة $SE(w_0, w_1)$):
$$\begin{array}{rcl} w_0^{(t+1)} = w_0^{(t)} -\eta \frac{\partial SE}{\partial w_0} |_{t} \\  w_1^{(t+1)} = w_1^{(t)} -\eta \frac{\partial SE}{\partial w_1} |_{t} \end{array}$$
وبحساب المشتقات الجزئية نحصل على ما يلي: 
$$\begin{array}{rcl} w_0^{(t+1)} = w_0^{(t)} + \eta \sum_{i=1}^{\ell}(y_i - w_0^{(t)} - w_1^{(t)}x_i) \\  w_1^{(t+1)} = w_1^{(t)} + \eta \sum_{i=1}^{\ell}(y_i - w_0^{(t)} - w_1^{(t)}x_i)x_i \end{array}$$
تعمل هذه الرياضيات بشكل جيد طالما أن كمية البيانات ليست كبيرة (لن نناقش المشكلات المتعلقة بالحد الأدنى المحلي ونقاط السرج واختيار معدل التعلم واللحظات والأشياء الأخرى - تمت تغطية هذه المواضيع بشكل شامل للغاية في [فصل الحساب الرقمي](http://www.deeplearningbook.org/contents/numerical.html) في "التعلم العميق"). 
هناك مشكلة تتعلق بالنسب التدرجي الدفعي - يتطلب تقييم التدرج جمع عدد من القيم لكل كائن من مجموعة التدريب. بمعنى آخر، تتطلب الخوارزمية الكثير من التكرارات، وكل تكرار يعيد حساب الأوزان باستخدام صيغة تحتوي على مجموع $\sum_{i=1}^\ell$ على مجموعة التدريب بأكملها. ماذا يحدث عندما يكون لدينا مليارات من عينات التدريب؟
<img src="@@KEEP_00046@@ />
ومن هنا الدافع للنزول التدرج العشوائي! ببساطة، نتخلص من علامة الجمع ونقوم بتحديث الأوزان فقط من خلال عينات تدريب فردية (أو عدد صغير منها). وفي حالتنا لدينا ما يلي:
$$\begin{array}{rcl} w_0^{(t+1)} = w_0^{(t)} + \eta (y_i - w_0^{(t)} - w_1^{(t)}x_i) \\  w_1^{(t+1)} = w_1^{(t)} + \eta (y_i - w_0^{(t)} - w_1^{(t)}x_i)x_i \end{array}$$مع هذا النهج، ليس هناك ما يضمن أننا سوف نتحرك في أفضل اتجاه ممكن في كل تكرار. لذلك، قد نحتاج إلى المزيد من التكرارات، ولكننا نحصل على تحديثات وزن أسرع بكثير.



لدى Andrew Ng مثال جيد على ذلك في [دورة التعلم الآلي](https://www.coursera.org/learn/machine-learning). دعونا نلقي نظرة.
<img src='@@KEEP_00048@@>
هذه هي المخططات الكنتورية لبعض الوظائف، ونريد العثور على الحد الأدنى الشامل لهذه الوظيفة. يوضح المنحنى الأحمر تغيرات الوزن (في هذه الصورة، $\theta_0$ و$\theta_1$ يتوافقان مع $w_0$ و$w_1$). وفقًا لخصائص التدرج، يكون اتجاه التغيير عند كل نقطة متعامدًا مع المخططات الكنتورية. مع النسب التدرج العشوائي، تتغير الأوزان بطريقة أقل قابلية للتنبؤ بها، بل قد يبدو أن بعض الخطوات خاطئة من خلال الابتعاد عن الحد الأدنى؛ ومع ذلك، كلا الإجراءين يتقاربان إلى نفس الحل.



### 1.2. النهج عبر الإنترنت للتعلم
يمنحنا النسب التدرج العشوائي إرشادات عملية لتدريب كل من المصنفين والتراجعيين بكميات كبيرة من البيانات تصل إلى مئات الجيجابايت (اعتمادًا على الموارد الحسابية).
بالنظر إلى حالة الانحدار المقترن، يمكننا تخزين مجموعة بيانات التدريب $(X,y)$ في محرك الأقراص الثابتة دون تحميلها في ذاكرة الوصول العشوائي (حيث لا تناسبها ببساطة)، وقراءة الكائنات واحدًا تلو الآخر، وتحديث أوزان نموذجنا:
$$\begin{array}{rcl} w_0^{(t+1)} = w_0^{(t)} + \eta (y_i - w_0^{(t)} - w_1^{(t)}x_i) \\  w_1^{(t+1)} = w_1^{(t)} + \eta (y_i - w_0^{(t)} - w_1^{(t)}x_i)x_i \end{array}$$
بعد العمل على مجموعة بيانات التدريب بأكملها، ستنخفض دالة الخسارة (على سبيل المثال، خطأ الجذر التربيعي في الانحدار أو الخسارة اللوجستية في التصنيف)، ولكن عادةً ما يستغرق الأمر عشرات التمريرات على مجموعة التدريب لجعل الخسارة صغيرة بما يكفي. 
يُطلق على هذا النهج في التعلم اسم **التعلم عبر الإنترنت**، وقد ظهر هذا الاسم حتى قبل أن يصبح التعلم الآلي MOOCs سائدًا.لم نناقش العديد من التفاصيل حول SGD هنا. إذا كنت تريد التعمق في النظرية، فإنني أوصي بشدة بـ ["Convex Optimization" بقلم ستيفن بويد](https://www.amazon.com/Convex-Optimization-Stephen-Boyd/dp/0521833787). الآن، سوف نقدم مكتبة Vowpal Wabbit، والتي تعد مفيدة لتدريب النماذج البسيطة بمجموعات بيانات ضخمة بفضل التحسين العشوائي وخدعة أخرى، وهي تجزئة الميزات.



في scikit-learn، تتم تسمية المصنفات والتراجعات المدربة باستخدام SGD `SGDClassifier` و`SGDRegressor` في `sklearn.linear_model`. هذه تطبيقات رائعة لـ SGD، لكننا سنركز على VW نظرًا لأنها أكثر أداءً من نماذج SGD الخاصة بـ sklearn في العديد من الجوانب.



## 2. معالجة الميزات الفئوية
###2.1. ترميز التسمية
تعمل العديد من خوارزميات التصنيف والانحدار في الفضاء الإقليدي أو المتري، مما يعني أن البيانات ممثلة بمتجهات من الأعداد الحقيقية. ومع ذلك، في البيانات الحقيقية، غالبًا ما يكون لدينا ميزات فئوية ذات قيم منفصلة مثل نعم/لا أو يناير/فبراير/.../ديسمبر. سنرى كيفية معالجة هذا النوع من البيانات، خاصة مع النماذج الخطية، وكيفية التعامل مع العديد من الميزات الفئوية حتى عندما تحتوي على العديد من القيم الفريدة.



دعنا نستكشف [مجموعة بيانات تسويق بنك UCI](https://archive.ics.uci.edu/ml/datasets/bank+marketing) حيث تكون معظم الميزات قاطعة.


In [ ]:
df = pd.read_csv(os.path.join(PATH_TO_ALL_DATA, "bank_train.csv"))
labels = pd.read_csv(
    os.path.join(PATH_TO_ALL_DATA, "bank_train_target.csv"), header=None
)

df.head()


يمكننا أن نرى أن معظم الميزات لا يتم تمثيلها بالأرقام. يمثل هذا مشكلة لأنه لا يمكننا استخدام معظم أساليب التعلم الآلي (على الأقل تلك المطبقة في scikit-learn) خارج الصندوق.
دعونا نتعمق في ميزة "التعليم".


In [ ]:
df["education"].value_counts().plot.barh();


الحل الأكثر وضوحًا هو تعيين كل قيمة لهذه الميزة إلى رقم فريد. على سبيل المثال، يمكننا تعيين `university.degree` إلى 0، و`basic.9y` إلى 1، وهكذا. يمكنك استخدام `sklearn.preprocessing.LabelEncoder` لإجراء هذا التعيين.


In [ ]:
label_encoder = LabelEncoder()

تبحث طريقة `fit` لهذه الفئة عن كافة القيم الفريدة وتقوم بإنشاء التعيين الفعلي بين الفئات والأرقام، وتقوم طريقة `transform` بتحويل الفئات إلى أرقام. بعد تنفيذ `fit`، سيكون لدى `label_encoder` السمة `classes_` مع كافة القيم الفريدة للميزة. دعونا نحسبها للتأكد من صحة التحويل.


In [ ]:
mapped_education = pd.Series(label_encoder.fit_transform(df["education"]))
mapped_education.value_counts().plot.barh()
print(dict(enumerate(label_encoder.classes_)))

In [ ]:
df["education"] = mapped_education
df.head()


لنطبق التحويل على أعمدة أخرى من النوع `object`.


In [ ]:
categorical_columns = df.columns[df.dtypes == "object"].union(["education"])
for column in categorical_columns:
    df[column] = label_encoder.fit_transform(df[column])
df.head()


المشكلة الرئيسية في هذا النهج هي أننا قدمنا الآن بعض الترتيب النسبي حيث قد لا يكون موجودًا.  
على سبيل المثال، قمنا بإدخال الجبر ضمنيًا على قيم ميزة الوظيفة حيث يمكننا الآن طرح وظيفة العميل رقم 2 من وظيفة العميل رقم 1:


In [ ]:
df.loc[1].job - df.loc[2].job


هل هذه العملية لها أي معنى؟ ليس حقيقيًا. دعونا نحاول تدريب الانحدار اللوجستي من خلال تحويل الميزة هذا.


In [ ]:
def logistic_regression_accuracy_on(dataframe, labels):
    features = dataframe.as_matrix()
    train_features, test_features, train_labels, test_labels = train_test_split(
        features, labels
    )

    logit = LogisticRegression()
    logit.fit(train_features, train_labels)
    return classification_report(test_labels, logit.predict(test_features))


print(logistic_regression_accuracy_on(df[categorical_columns], labels))


يمكننا أن نرى أن الانحدار اللوجستي لا يتنبأ أبدًا بالفئة 1. ومن أجل استخدام النماذج الخطية ذات الميزات الفئوية، سنستخدم نهجًا مختلفًا: التشفير الساخن الواحد.
###2.2. ترميز واحد ساخن
لنفترض أن بعض الميزات يمكن أن تحتوي على واحدة من 10 قيم فريدة. ينشئ التشفير السريع 10 ميزات جديدة تتوافق مع هذه القيم الفريدة، جميعها *ما عدا واحدة* هي أصفار.


In [ ]:
one_hot_example = pd.DataFrame([{i: 0 for i in range(10)}])
one_hot_example.loc[0, 6] = 1
one_hot_example


تم تنفيذ هذه الفكرة في فئة `OneHotEncoder` من `sklearn.preprocessing`. افتراضيًا، `OneHotEncoder` يحول البيانات إلى مصفوفة متفرقة لتوفير مساحة الذاكرة لأن معظم القيم هي أصفار ولأننا لا نريد استهلاك المزيد من ذاكرة الوصول العشوائي (RAM). ومع ذلك، في هذا المثال تحديدًا، لم نواجه مثل هذه المشكلات، لذلك سنستخدم تمثيل المصفوفة "الكثيف".


In [ ]:
onehot_encoder = OneHotEncoder(sparse=False)

In [ ]:
encoded_categorical_columns = pd.DataFrame(
    onehot_encoder.fit_transform(df[categorical_columns])
)
encoded_categorical_columns.head()

لدينا 53 عمودًا تتوافق مع عدد القيم الفريدة للميزات الفئوية في مجموعة البيانات الخاصة بنا. عند التحويل باستخدام One-Hot Encoding، يمكن استخدام هذه البيانات مع النماذج الخطية:


In [ ]:
print(logistic_regression_accuracy_on(encoded_categorical_columns, labels))


###2.3. خدعة التجزئة
يمكن أن تكون البيانات الحقيقية متقلبة، مما يعني أننا لا نستطيع ضمان عدم ظهور قيم جديدة للميزات الفئوية. تعيق هذه المشكلة استخدام نموذج مدرب على البيانات الجديدة. بالإضافة إلى ذلك، `LabelEncoder` يتطلب تحليلًا أوليًا لمجموعة البيانات بأكملها وتخزين التعيينات التي تم إنشاؤها في الذاكرة، مما يجعل من الصعب العمل مع مجموعات البيانات الكبيرة.
هناك طريقة بسيطة لتوجيه البيانات الفئوية بناءً على التجزئة وتُعرف باسم خدعة التجزئة، وليس من المستغرب أن تكون هذه الحيلة. 
يمكن أن تساعدنا وظائف التجزئة في العثور على رموز فريدة لقيم ميزات مختلفة، على سبيل المثال:


In [ ]:
for s in ("university.degree", "high.school", "illiterate"):
    print(s, "->", hash(s))


لن نستخدم قيمًا سالبة أو قيمًا ذات حجم كبير، لذلك نقوم بتقييد نطاق قيم دالة التجزئة:


In [ ]:
hash_space = 25
for s in ("university.degree", "high.school", "illiterate"):
    print(s, "->", hash(s) % hash_space)


تخيل أن مجموعة البيانات لدينا تحتوي على طالب واحد (أي غير متزوج)، تلقى مكالمة هاتفية يوم الاثنين. سيتم إنشاء متجهات الميزات الخاصة به بشكل مشابه كما في حالة One-Hot Encoding ولكن في مساحة ذات نطاق ثابت لجميع الميزات:


In [ ]:
hashing_example = pd.DataFrame([{i: 0.0 for i in range(hash_space)}])
for s in ("job=student", "marital=single", "day_of_week=mon"):
    print(s, "->", hash(s) % hash_space)
    hashing_example.loc[0, hash(s) % hash_space] = 1
hashing_example


نريد أن نشير إلى أننا لا نقوم بتجزئة قيم الميزات فحسب، بل أيضًا أزواج من **اسم الميزة + قيمة الميزة**. من المهم القيام بذلك حتى نتمكن من التمييز بين نفس القيم والميزات المختلفة.


In [ ]:
assert hash("no") == hash("no")
assert hash("housing=no") != hash("loan=no")


هل من الممكن حدوث تصادم عند استخدام رموز التجزئة؟ بالتأكيد، هذا ممكن، لكنه حالة نادرة تحتوي على مساحات تجزئة كبيرة بما يكفي. وحتى في حالة حدوث تصادم، فإن مقاييس الانحدار أو التصنيف لن تعاني كثيرًا. في هذه الحالة، تعمل تصادمات التجزئة كشكل من أشكال التنظيم.
<img src="@@KEEP_00051@@>ربما تقول "WTF؟"؛ يبدو التجزئة غير بديهي. هذا صحيح، لكن هذه الاستدلالات تكون في بعض الأحيان، في الواقع، الطريقة الوحيدة المعقولة للتعامل مع البيانات الفئوية (ماذا يمكنك أن تفعل أيضًا إذا كان لديك 30 مليون ميزة؟). علاوة على ذلك، أثبتت هذه التقنية فعاليتها. ومع تعاملك بشكل أكبر مع البيانات، قد ترى ذلك بنفسك.
تم إجراء تحليل جيد لتصادمات التجزئة واعتمادها على مساحة الميزات وأبعاد مساحة التجزئة والتأثير على أداء التصنيف/الانحدار في [هذه المقالة](https://booking.ai/dont-be-tricked-by-the-hashing-trick-192a6aae3087) بواسطة Booking.com. 



## 3. فاوبال وابيت



[Vowpal Wabbit](https://github.com/JohnLangford/vowpal_wabbit) (VW) هي إحدى مكتبات التعلم الآلي الأكثر انتشارًا والمستخدمة في الصناعة. ويتميز بسرعة التدريب ودعمه للعديد من أوضاع التدريب، خاصة للتعلم عبر الإنترنت باستخدام البيانات الكبيرة وعالية الأبعاد. وهذه واحدة من المزايا الرئيسية للمكتبة. أيضًا، مع تنفيذ خدعة التجزئة، يعد Vowpal Wabbit خيارًا مثاليًا للعمل مع البيانات النصية.
Shell هي الواجهة الرئيسية لشركة VW.


In [ ]:
!vw --help


يقرأ Vowpal Wabbit البيانات من الملفات أو من دفق الإدخال القياسي (stdin) بالتنسيق التالي:
`[Label] [Importance] [Tag]|Namespace Features |Namespace Features ... |Namespace Features`
`Namespace=String[:Value]`
`Features=(String[:Value] )*`
هنا يشير [] إلى عناصر غير إلزامية، و(...)\* يعني السماح بإدخالات متعددة.- **العلامة** هي رقم. في حالة التصنيف، عادة ما يكون 1 و -1؛ بالنسبة للانحدار، فهي قيمة تعويم حقيقية
- **الأهمية** هي رقم. ويدل على وزن العينة أثناء التدريب. يساعد إعداد هذا عند العمل مع البيانات غير المتوازنة.
- **العلامة** عبارة عن سلسلة بدون مسافات. إنه "اسم" العينة الذي تحفظه شركة فولكس فاجن عند التنبؤ. لفصل العلامة عن الأهمية، من الأفضل أن تبدأ العلامة بالحرف '.
- **مساحة الاسم** مخصصة لإنشاء مساحات ميزات مختلفة. 
- **الميزات** هي ميزات الكائن داخل **مساحة الاسم** المحددة. الميزات لها وزن 1.0 بشكل افتراضي، ولكن يمكن تغييره، على سبيل المثال الميزة:0.1. 
السلسلة التالية تطابق تنسيق VW:
```
1 1.0 |Subject WHAT car is this |Organization University of Maryland:0.5 College Park
```
دعونا نتحقق من التنسيق عن طريق تشغيل VW باستخدام نموذج التدريب هذا:


In [ ]:
! echo '1 1.0 |Subject WHAT car is this |Organization University of Maryland:0.5 College Park' | vw


تعد VW أداة رائعة للعمل مع البيانات النصية. سنوضح ذلك باستخدام [مجموعة بيانات 20newsgroups](http://scikit-learn.org/stable/datasets/twenty_newsgroups.html)، والتي تحتوي على رسائل من 20 رسالة إخبارية مختلفة.
###3.1. أخبار. التصنيف الثنائي.


In [ ]:
# load data with sklearn's function
newsgroups = fetch_20newsgroups(PATH_TO_ALL_DATA)

In [ ]:
newsgroups["target_names"]


لنلقِ نظرة على الوثيقة الأولى في هذه المجموعة:


In [ ]:
text = newsgroups["data"][0]
target = newsgroups["target_names"][newsgroups["target"][0]]

print("-----")
print(target)
print("-----")
print(text.strip())
print("----")


نقوم الآن بتحويل البيانات إلى شيء يستطيع Vowpal Wabbit فهمه. سوف نقوم برمي الكلمات الأقصر من 3 رموز. هنا، سوف نتخطى بعض مراحل البرمجة اللغوية العصبية المهمة مثل الاشتقاق والتجسيد؛ ولكننا سنرى لاحقاً أن شركة فولكس فاجن تحل المشكلة حتى بدون هذه الخطوات.


In [ ]:
def to_vw_format(document, label=None):
    return (
        str(label or "")
        + " |text "
        + " ".join(re.findall("\w{3,}", document.lower()))
        + "\n"
    )


to_vw_format(text, 1 if target == "rec.autos" else -1)


قمنا بتقسيم مجموعة البيانات إلى تدريب واختبار وكتابتها في ملفات منفصلة. سنعتبر المستند إيجابيًا إذا كان يتوافق مع **rec.autos**. وهكذا نقوم ببناء نموذج يميز المقالات المتعلقة بالسيارات عن المواضيع الأخرى: 


In [ ]:
all_documents = newsgroups["data"]
all_targets = [
    1 if newsgroups["target_names"][target] == "rec.autos" else -1
    for target in newsgroups["target"]
]

In [ ]:
train_documents, test_documents, train_labels, test_labels = train_test_split(
    all_documents, all_targets, random_state=7
)

with open(os.path.join(PATH_TO_ALL_DATA, "20news_train.vw"), "w") as vw_train_data:
    for text, target in zip(train_documents, train_labels):
        vw_train_data.write(to_vw_format(text, target))
with open(os.path.join(PATH_TO_ALL_DATA, "20news_test.vw"), "w") as vw_test_data:
    for text in test_documents:
        vw_test_data.write(to_vw_format(text))

الآن، نقوم بتمرير ملف التدريب الذي تم إنشاؤه إلى Vowpal Wabbit. لقد قمنا بحل مشكلة التصنيف باستخدام دالة فقدان المفصلة (SVM الخطية). سيتم حفظ النموذج المدرب في الملف `20news_model.vw`:


In [ ]:
!vw -d $PATH_TO_ALL_DATA/20news_train.vw \
 --loss_function hinge -f $PATH_TO_ALL_DATA/20news_model.vw


تقوم شركة VW بطباعة الكثير من المعلومات المثيرة للاهتمام أثناء التدريب (يمكن للمرء منعها باستخدام المعلمة `--quiet`). يمكنك الاطلاع على وثائق المخرجات التشخيصية على [GitHub](https://github.com/JohnLangford/vowpal_wabbit/wiki/Tutorial#vws-diagnostic-information). لاحظ كيف ينخفض ​​متوسط ​​الخسارة أثناء التدريب. بالنسبة لحساب الخسارة، تستخدم شركة فولكس فاجن عينات لم ترها من قبل، لذلك عادة ما يكون هذا القياس دقيقًا. الآن، نطبق نموذجنا المدرب على مجموعة الاختبار، ونحفظ التوقعات في ملف يحمل العلامة `-p`:  


In [ ]:
!vw -i $PATH_TO_ALL_DATA/20news_model.vw -t -d $PATH_TO_ALL_DATA/20news_test.vw \
-p $PATH_TO_ALL_DATA/20news_test_predictions.txt


نقوم الآن بتحميل توقعاتنا، وحساب AUC، ورسم منحنى ROC:


In [ ]:
with open(os.path.join(PATH_TO_ALL_DATA, "20news_test_predictions.txt")) as pred_file:
    test_prediction = [float(label) for label in pred_file.readlines()]

auc = roc_auc_score(test_labels, test_prediction)
roc_curve = roc_curve(test_labels, test_prediction)

with plt.xkcd():
    plt.plot(roc_curve[0], roc_curve[1])
    plt.plot([0, 1], [0, 1])
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title("test AUC = %f" % (auc))
    plt.axis([-0.05, 1.05, -0.05, 1.05]);


تظهر قيمة الجامعة الأمريكية بالقاهرة التي حصلنا عليها أننا حققنا جودة تصنيف عالية.



###3.2. أخبار. تصنيف متعدد الطبقات



سنستخدم نفس مجموعة بيانات الأخبار، لكن هذه المرة سنحل مشكلة التصنيف متعدد الفئات. `Vowpal Wabbit` صعب الإرضاء بعض الشيء – فهو يريد تسميات تبدأ من 1 حتى K، حيث K – هو عدد الفئات في مهمة التصنيف (20 في حالتنا). لذلك سوف نستخدم LabelEncoder ونضيف 1 بعد ذلك (تذكر أن `LabelEncoder` يعين التصنيفات في النطاق من 0 إلى K-1).


In [ ]:
all_documents = newsgroups["data"]
topic_encoder = LabelEncoder()
all_targets_mult = topic_encoder.fit_transform(newsgroups["target"]) + 1


**البيانات هي نفسها، ولكننا قمنا بتغيير التسميات، Train_labels_mult وtest_labels_mult، إلى متجهات التسميات من 1 إلى 20.**


In [ ]:
train_documents, test_documents, train_labels_mult, test_labels_mult = train_test_split(
    all_documents, all_targets_mult, random_state=7
)

with open(os.path.join(PATH_TO_ALL_DATA, "20news_train_mult.vw"), "w") as vw_train_data:
    for text, target in zip(train_documents, train_labels_mult):
        vw_train_data.write(to_vw_format(text, target))
with open(os.path.join(PATH_TO_ALL_DATA, "20news_test_mult.vw"), "w") as vw_test_data:
    for text in test_documents:
        vw_test_data.write(to_vw_format(text))

نحن ندرب Vowpal Wabbit في وضع التصنيف متعدد الفئات، ونمرر المعلمة `oaa`("واحد مقابل الكل") مع عدد الفئات. دعونا نرى أيضًا المعلمات التي تعتمد عليها جودة نموذجنا (يمكن العثور على مزيد من المعلومات في [البرنامج التعليمي الرسمي لـ Vowpal Wabbit](https://github.com/JohnLangford/vowpal_wabbit/wiki/Tutorial)):
 - معدل التعلم (-l، 0.5 افتراضي) - معدل تغير الوزن في كل خطوة
 - تضاؤل معدل التعلم (--power_t, 0.5 default) - ثبت عمليًا أنه إذا انخفض معدل التعلم مع عدد خطوات نزول التدرج العشوائي، فإننا نقترب من الحد الأدنى من الخسارة بشكل أفضل
 - دالة الخسارة (-loss_function) - تعتمد عليها خوارزمية التدريب بأكملها. راجع [docs](https://github.com/JohnLangford/vowpal_wabbit/wiki/Loss-functions) للتعرف على وظائف الخسارة
 - التنظيم (-l1) - لاحظ أن شركة VW تقوم بحساب التنظيم لكل كائن. ولهذا السبب نقوم عادةً بتعيين قيم التنظيم على حوالي $10^{-20}.$
 
بالإضافة إلى ذلك، يمكننا تجربة الضبط التلقائي لمعلمات Vowpal Wabbit باستخدام [Hyperopt](https://github.com/hyperopt/hyperopt).


In [ ]:
%%time
!vw --oaa 20 $PATH_TO_ALL_DATA/20news_train_mult.vw -f $PATH_TO_ALL_DATA/20news_model_mult.vw \
--loss_function=hinge

In [ ]:
%%time
!vw -i $PATH_TO_ALL_DATA/20news_model_mult.vw -t -d $PATH_TO_ALL_DATA/20news_test_mult.vw \
-p $PATH_TO_ALL_DATA/20news_test_predictions_mult.txt

In [ ]:
with open(
    os.path.join(PATH_TO_ALL_DATA, "20news_test_predictions_mult.txt")
) as pred_file:
    test_prediction_mult = [float(label) for label in pred_file.readlines()]

In [ ]:
accuracy_score(test_labels_mult, test_prediction_mult)


إليك عدد المرات التي يخطئ فيها النموذج في تصنيف الإلحاد مع موضوعات أخرى:


In [ ]:
M = confusion_matrix(test_labels_mult, test_prediction_mult)
for i in np.where(M[0, :] > 0)[0][1:]:
    print(newsgroups["target_names"][i], M[0, i])


###3.3. تقييمات الأفلام على موقع IMDB
في هذا الجزء سنقوم بتصنيف ثنائي لمراجعات الأفلام [IMDB](http://www.imdb.com) (قاعدة بيانات الأفلام الدولية). سنرى مدى سرعة أداء Vowpal Wabbit.
باستخدام وظيفة `load_files` من `sklearn.datasets`، نقوم بتحميل مجموعات بيانات مراجعات الأفلام. إنها نفس مجموعة البيانات التي استخدمناها في دفتر الملاحظات subject04 Part4.


In [ ]:
import tarfile
# Download the dataset if not already in place
from io import BytesIO

import requests

url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"


def load_imdb_dataset(extract_path="../../data", overwrite=False):
    # check if existed already
    if (
        os.path.isfile(os.path.join(extract_path, "aclImdb", "README"))
        and not overwrite
    ):
        print("IMDB dataset is already in place.")
        return

    print("Downloading the dataset from:  ", url)
    response = requests.get(url)

    tar = tarfile.open(mode="r:gz", fileobj=BytesIO(response.content))

    data = tar.extractall(extract_path)


load_imdb_dataset()


قراءة بيانات القطار، تسميات منفصلة.


In [ ]:
PATH_TO_IMDB = "../../data/aclImdb"

reviews_train = load_files(
    os.path.join(PATH_TO_IMDB, "train"), categories=["pos", "neg"]
)

text_train, y_train = reviews_train.data, reviews_train.target

In [ ]:
print("Number of documents in training data: %d" % len(text_train))
print(np.bincount(y_train))


افعل الشيء نفسه بالنسبة لمجموعة الاختبار.


In [ ]:
reviews_test = load_files(os.path.join(PATH_TO_IMDB, "test"), categories=["pos", "neg"])
text_test, y_test = reviews_test.data, reviews_test.target

In [ ]:
print("Number of documents in test data: %d" % len(text_test))
print(np.bincount(y_test))


ألق نظرة على أمثلة المراجعات والتسميات المقابلة لها.


In [ ]:
text_train[0]

In [ ]:
y_train[0]  # good review

In [ ]:
text_train[1]

In [ ]:
y_train[1]  # bad review

In [ ]:
to_vw_format(str(text_train[1]), 1 if y_train[0] == 1 else -1)


الآن، نقوم بإعداد التدريب (`movie_reviews_train.vw`)، والتحقق من الصحة (`movie_reviews_valid.vw`)، واختبار (`movie_reviews_test.vw`) لـ Vowpal Wabbit. سوف نستخدم 70% للتدريب، و30% لمجموعة الإيقاف.


In [ ]:
train_share = int(0.7 * len(text_train))
train, valid = text_train[:train_share], text_train[train_share:]
train_labels, valid_labels = y_train[:train_share], y_train[train_share:]

In [ ]:
len(train_labels), len(valid_labels)

In [ ]:
with open(
    os.path.join(PATH_TO_ALL_DATA, "movie_reviews_train.vw"), "w"
) as vw_train_data:
    for text, target in zip(train, train_labels):
        vw_train_data.write(to_vw_format(str(text), 1 if target == 1 else -1))
with open(
    os.path.join(PATH_TO_ALL_DATA, "movie_reviews_valid.vw"), "w"
) as vw_train_data:
    for text, target in zip(valid, valid_labels):
        vw_train_data.write(to_vw_format(str(text), 1 if target == 1 else -1))
with open(os.path.join(PATH_TO_ALL_DATA, "movie_reviews_test.vw"), "w") as vw_test_data:
    for text in text_test:
        vw_test_data.write(to_vw_format(str(text)))

In [ ]:
!head -2 $PATH_TO_ALL_DATA/movie_reviews_train.vw

In [ ]:
!head -2 $PATH_TO_ALL_DATA/movie_reviews_valid.vw

In [ ]:
!head -2 $PATH_TO_ALL_DATA/movie_reviews_test.vw

**الآن نقوم بتشغيل Vowpal Wabbit بالوسيطات التالية:**
 - `-d`، المسار إلى مجموعة التدريب (ملف .vw المقابل)
 - `--loss_function` - المفصلة (لا تتردد في التجربة هنا)
 - `-f` - المسار إلى ملف الإخراج (والذي يمكن أن يكون أيضًا بتنسيق .vw)


In [ ]:
!vw -d $PATH_TO_ALL_DATA/movie_reviews_train.vw --loss_function hinge \
-f $PATH_TO_ALL_DATA/movie_reviews_model.vw --quiet


بعد ذلك، قم بالتنبؤ بالتوقف باستخدام وسيطات VW التالية:
 - `-i` – المسار إلى النموذج المدرب (ملف .vw)
 - `-d` - المسار إلى مجموعة الإيقاف (ملف .vw) 
 - `-p` - المسار إلى ملف txt حيث سيتم تخزين التوقعات
 - `-t` - يطلب من شركة VW تجاهل الملصقات


In [ ]:
!vw -i $PATH_TO_ALL_DATA/movie_reviews_model.vw -t \
-d $PATH_TO_ALL_DATA/movie_reviews_valid.vw -p $PATH_TO_ALL_DATA/movie_valid_pred.txt --quiet


اقرأ التوقعات من الملف النصي وقم بتقدير الدقة وROC AUC. لاحظ أن شركة VW تطبع تقديرات الاحتمالية لفئة +1. يتم توزيع هذه التقديرات من -1 إلى 1، حتى نتمكن من تحويلها إلى إجابات ثنائية، على افتراض أن القيم الإيجابية تنتمي إلى الفئة 1.


In [ ]:
with open(os.path.join(PATH_TO_ALL_DATA, "movie_valid_pred.txt")) as pred_file:
    valid_prediction = [float(label) for label in pred_file.readlines()]
print(
    "Accuracy: {}".format(
        round(
            accuracy_score(
                valid_labels, [int(pred_prob > 0) for pred_prob in valid_prediction]
            ),
            3,
        )
    )
)
print("AUC: {}".format(round(roc_auc_score(valid_labels, valid_prediction), 3)))


مرة أخرى، افعل نفس الشيء مع مجموعة الاختبار.


In [ ]:
!vw -i $PATH_TO_ALL_DATA/movie_reviews_model.vw -t \
-d $PATH_TO_ALL_DATA/movie_reviews_test.vw \
-p $PATH_TO_ALL_DATA/movie_test_pred.txt --quiet

In [ ]:
with open(os.path.join(PATH_TO_ALL_DATA, "movie_test_pred.txt")) as pred_file:
    test_prediction = [float(label) for label in pred_file.readlines()]
print(
    "Accuracy: {}".format(
        round(
            accuracy_score(
                y_test, [int(pred_prob > 0) for pred_prob in test_prediction]
            ),
            3,
        )
    )
)
print("AUC: {}".format(round(roc_auc_score(y_test, test_prediction), 3)))


دعونا نحاول تحقيق دقة أعلى من خلال دمج الصور الكبيرة.


In [ ]:
!vw -d $PATH_TO_ALL_DATA/movie_reviews_train.vw \
--loss_function hinge --ngram 2 -f $PATH_TO_ALL_DATA/movie_reviews_model2.vw --quiet

In [ ]:
!vw -i$PATH_TO_ALL_DATA/movie_reviews_model2.vw -t -d $PATH_TO_ALL_DATA/movie_reviews_valid.vw \
-p $PATH_TO_ALL_DATA/movie_valid_pred2.txt --quiet

In [ ]:
with open(os.path.join(PATH_TO_ALL_DATA, "movie_valid_pred2.txt")) as pred_file:
    valid_prediction = [float(label) for label in pred_file.readlines()]
print(
    "Accuracy: {}".format(
        round(
            accuracy_score(
                valid_labels, [int(pred_prob > 0) for pred_prob in valid_prediction]
            ),
            3,
        )
    )
)
print("AUC: {}".format(round(roc_auc_score(valid_labels, valid_prediction), 3)))

In [ ]:
!vw -i $PATH_TO_ALL_DATA/movie_reviews_model2.vw -t -d $PATH_TO_ALL_DATA/movie_reviews_test.vw \
-p $PATH_TO_ALL_DATA/movie_test_pred2.txt --quiet

In [ ]:
with open(os.path.join(PATH_TO_ALL_DATA, "movie_test_pred2.txt")) as pred_file:
    test_prediction2 = [float(label) for label in pred_file.readlines()]
print(
    "Accuracy: {}".format(
        round(
            accuracy_score(
                y_test, [int(pred_prob > 0) for pred_prob in test_prediction2]
            ),
            3,
        )
    )
)
print("AUC: {}".format(round(roc_auc_score(y_test, test_prediction2), 3)))


لقد ساعدت إضافة الصور الكبيرة حقًا في تحسين نموذجنا!



###3.4. تصنيف غيغابايت من أسئلة StackOverflow



تم نقل هذا القسم إلى Kaggle، يرجى استكشاف [هذه النواة](https://www.kaggle.com/kashnitsky/topic-8-online-learning-and-vowpal-wabbit).



## 4. مهمة تجريبية
لفهم التعلم العشوائي بشكل أفضل، يمكنك إكمال [هذه المهمة](https://www.kaggle.com/kashnitsky/assignment-8-implementing-online-regressor) حيث سيُطلب منك تنفيذ تراجع التدرج العشوائي من البداية. هذه المهمة مخصصة لك فقط للتدرب عليها، وتأتي مع [الحل](https://www.kaggle.com/kashnitsky/a8-demo-implementing-online-regressor-solution).## 5. موارد مفيدة
- نفس دفتر الملاحظات التفاعلي القائم على الويب [Kaggle Kernel](https://www.kaggle.com/kashnitsky/topic-8-online-learning-and-vowpal-wabbit)
- ["التدريب أثناء القراءة"](https://www.kaggle.com/kashnitsky/training-while-reading-vowpal-wabbit-starter) - مثال على استخدام غلاف بايثون
- الطبق الرئيسي [الموقع](https://mlcourse.ai)، [مستودع الدورة](https://github.com/Yorko/mlcourse.ai)، ويوتيوب [القناة](https://www.youtube.com/watch?v=QKTuw4PNOsU&list=PLVlY_7IJCMJeRfZ68eVfEcu-UcN9BbwiX)
- مواد الدورة التدريبية باعتبارها [مجموعة بيانات Kaggle](https://www.kaggle.com/kashnitsky/mlcourse)
- [الوثائق] الرسمية لشركة فولكس فاجن (https://github.com/JohnLangford/vowpal_wabbit/wiki) على جيثب
- ["Vowpal Wabbit رائع"](https://github.com/VowpalWabbit/vowpal_wabbit/wiki/Awesome-Vowpal-Wabbit) Wiki
- [لا تنخدع بخدعة التجزئة](https://booking.ai/dont-be-tricked-by-the-hashing-trick-192a6aae3087) - تحليل تصادمات التجزئة واعتمادها على مساحة الميزات وأبعاد مساحة التجزئة والتأثير على أداء التصنيف/الانحدار
- [فصل "الحساب الرقمي"](http://www.deeplearningbook.org/contents/numerical.html) من [كتاب التعلم العميق](http://www.deeplearningbook.org/)
- ["التحسين المحدب" بقلم ستيفن بويد](https://www.amazon.com/Convex-Optimization-Stephen-Boyd/dp/0521833787)
- "يمكن أن تكون أدوات سطر الأوامر أسرع بمقدار 235 مرة من مجموعة Hadoop الخاصة بك" [منشور](https://aadrake.com/command-line-tools-can-be-235x-faster-than-your-hadoop-cluster.html)
- مقارنة خوارزميات تعلم الآلة المختلفة على مجموعة بيانات Criteo 1TB على [GitHub](https://github.com/rambler-digital-solutions/criteo-1tb-benchmark)
- [VW على FastML.com](http://fastml.com/blog/categories/vw/)